## Setup

First, let's set up the Python environment and import necessary libraries.

In [1]:
import sys
from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from bicep.analysis import BicepResults
from bicep.tech_adoption import TechnologyAdoption

print("Environment setup complete!")

Environment setup complete!


## Data Sources Used in BICEP

BICEP uses illustrative datasets from several open-source models and frameworks. This documentation uses these illustrative datasets to demonstrate BICEP functionality. **The BICEP tool is intended to be run on any arbitrary technology time series projection.** If you want to use your own technology forecast, you can bring your own.

### Building Stock Data Sources

**BICEP uses data from ResStock and ComStock models:**

- [ResStock](https://www.nrel.gov/buildings/resstock.html) - Residential building energy consumption estimates for ~880,000 representative US houses
- [ComStock](https://www.nrel.gov/buildings/comstock.html) - Commercial building energy consumption estimates for ~180,000 representative US commercial buildings

These models provide detailed building characteristics including:
- Building type, age, size, location
- End-use consumption profiles (heating, cooling, water heating, etc.)
- Peak electrical load estimates

### Technology Adoption Forecast Sources

**For our illustrative scenarios, we use forecasts from:**

1. **Scout Model** - Building sector technology adoption forecasts for heat pumps and heat pump water heaters
2. **ReEDS Model** - Electricity sector solar photovoltaic capacity forecasts by region
3. **TEMPO Model** - Transportation sector electric vehicle adoption projections

**Two Scenarios:**
- **BAU (Business-As-Usual):** Conservative technology adoption aligned with stated policies
- **High Electrification:** Accelerated technology adoption for aggressive climate goals

### Minimum Data Requirements

**To use BICEP with your own technology forecasts, you only need:**

- **Base Year (e.g., 2020):** Stock count for each technology
- **End Year (e.g., 2050):** Projected stock count for each technology
- **Geographic aggregation level:** State, utility territory, or custom region

BICEP will interpolate between base and end years to create adoption trajectories.

## About These Datasets

**Illustrative Data Notice:** The datasets and technology adoption forecasts presented in this documentation are for illustrative purposes only and represent a snapshot of model outputs. The data is not updated in real-time and should not be used for actual planning without verification.

**TEMPO Dataset Note:** For electric vehicle charging infrastructure, we reference projections from the [TEMPO Project](https://www.nrel.gov/transportation/tempo.html), which provides county-level EV charging demand estimates based on vehicle adoption forecasts.

**Key Characteristics of BICEP Datasets:**

- Data spans 2020-2050 to capture long-term technology adoption trends
- Building-level analysis with aggregation to state level
- Covers residential, commercial, and public EV charging sectors
- Probabilistic cost estimates reflecting uncertainty in upgrade requirements

In [4]:
# Create a technologies overview
technologies = {
    'Technology': [
        'Electric Vehicles (EV)',
        'Heat Pumps (HP)',
        'Heat Pump Water Heaters (HPWH)',
        'Solar Photovoltaic (PV)'
    ],
    'End Use': [
        'Transportation',
        'Space Heating/Cooling',
        'Water Heating',
        'Electricity Generation'
    ],
    'Typical Amperage': [
        '50 A (Level 2 charger)',
        '30-60 A',
        '20-30 A',
        'Variable (reduces net load)'
    ],
    'Scenarios Available': [
        'BAU, High',
        'BAU, High',
        'BAU, High',
        'BAU, High'
    ]
}

tech_df = pd.DataFrame(technologies)
print("Technologies Analyzed by BICEP:")
print(tech_df.to_string(index=False))

Technologies Analyzed by BICEP:
                    Technology                End Use            Typical Amperage Scenarios Available
        Electric Vehicles (EV)         Transportation      50 A (Level 2 charger)           BAU, High
               Heat Pumps (HP)  Space Heating/Cooling                     30-60 A           BAU, High
Heat Pump Water Heaters (HPWH)          Water Heating                     20-30 A           BAU, High
       Solar Photovoltaic (PV) Electricity Generation Variable (reduces net load)           BAU, High


## Technology Adoption Across Scenarios

Let's compare how technology adoption differs between the BAU (Business As Usual) and High electrification scenarios.

## Example Technology Adoption Forecasts

Below are examples of what technology adoption forecasts look like in BICEP. These illustrative forecasts show the typical trajectory from a base year to an end year.


In [2]:
# Create example technology adoption forecast visualization
years = np.array([2020, 2025, 2030, 2035, 2040, 2045, 2050])

# BAU scenario - conservative adoption
bau_ev = np.array([2, 5, 12, 22, 35, 48, 60]) / 100  # percentage of buildings
bau_hp = np.array([8, 12, 18, 25, 35, 45, 55]) / 100
bau_hpwh = np.array([5, 8, 12, 18, 25, 33, 42]) / 100
bau_pv = np.array([3, 6, 12, 20, 30, 42, 55]) / 100

# High scenario - aggressive adoption
high_ev = np.array([3, 10, 25, 40, 55, 70, 80]) / 100
high_hp = np.array([10, 18, 32, 48, 62, 75, 85]) / 100
high_hpwh = np.array([7, 15, 28, 42, 55, 68, 78]) / 100
high_pv = np.array([5, 12, 28, 45, 62, 75, 85]) / 100

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Electric Vehicles', 'Heat Pumps', 'Heat Pump Water Heaters', 'Solar PV'),
    specs=[[{}, {}], [{}, {}]]
)

# EV adoption
fig.add_trace(go.Scatter(x=years, y=bau_ev, name='BAU (EV)', mode='lines+markers', 
                          line=dict(color='#f59e0b', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=years, y=high_ev, name='High (EV)', mode='lines+markers',
                          line=dict(color='#d97706', width=2, dash='dash')), row=1, col=1)

# Heat Pump adoption
fig.add_trace(go.Scatter(x=years, y=bau_hp, name='BAU (HP)', mode='lines+markers',
                          line=dict(color='#f59e0b', width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=years, y=high_hp, name='High (HP)', mode='lines+markers',
                          line=dict(color='#d97706', width=2, dash='dash'), showlegend=False), row=1, col=2)

# Heat Pump Water Heater adoption
fig.add_trace(go.Scatter(x=years, y=bau_hpwh, name='BAU (HPWH)', mode='lines+markers',
                          line=dict(color='#f59e0b', width=2), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=years, y=high_hpwh, name='High (HPWH)', mode='lines+markers',
                          line=dict(color='#d97706', width=2, dash='dash'), showlegend=False), row=2, col=1)

# Solar PV adoption
fig.add_trace(go.Scatter(x=years, y=bau_pv, name='BAU (PV)', mode='lines+markers',
                          line=dict(color='#f59e0b', width=2), showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=years, y=high_pv, name='High (PV)', mode='lines+markers',
                          line=dict(color='#d97706', width=2, dash='dash'), showlegend=False), row=2, col=2)

# Update y-axes
fig.update_yaxes(title_text="Adoption Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Adoption Rate (%)", row=1, col=2)
fig.update_yaxes(title_text="Adoption Rate (%)", row=2, col=1)
fig.update_yaxes(title_text="Adoption Rate (%)", row=2, col=2)

# Update x-axes
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(title_text="Year", row=i, col=j)

fig.update_layout(height=700, width=1000, 
                  title_text="<b>Example Technology Adoption Forecasts</b><br><sub>Illustrative scenarios showing BAU vs High electrification</sub>",
                  hovermode='x unified')
fig.show()

print("\nThese adoption trajectories represent the % of buildings with each technology installed.")
print("The gap between BAU and High scenarios shows the impact of policy and market conditions.")



These adoption trajectories represent the % of buildings with each technology installed.
The gap between BAU and High scenarios shows the impact of policy and market conditions.


In [5]:
# Load real BICEP results data from CSV files
data_path = bicep_root / 'data' / 'parsed_inputs'

# Load BAU results
bau_res = pd.read_csv(data_path / 'bicep_results_bau_all_states.csv')

# Load High scenario results
high_res = pd.read_csv(data_path / 'bicep_results_high_all_states.csv')

print(f"BAU buildings: {len(bau_res):,}")
print(f"High buildings: {len(high_res):,}")
print(f"\nBAU columns:")
print(list(bau_res.columns))
print(f"\nBAU data sample:")
print(bau_res.head())

BAU buildings: 884,596
High buildings: 884,596

BAU columns:
['building_id', 'residential', 'max_elec_consumption_kwh', 'timestamp', 'upgrade', 'state', 'file_path', 'release', 'sqft', 'weight', 'total_units', 'peak_kw', 'peak_amp', 'assumed_volt', 'utilization', 'est_capacity', 'req_capacity', 'installed_capacity', 'spare_capacity', 'hp_peak_diff_kwh', 'hpwh_peak_diff_kwh', 'hp_req_capacity_amp', 'hpwh_req_capacity_amp', 'pv_relative_size', 'pv_size_kw', 'pv_req_capacity_amp', 'total_parking_spaces', 'perc_ev_spaces', 'ev_spaces', 'represented_vehicles', 'ev_req_capacity_amp', 'ev_adopted', 'pv_adopted', 'hp_adopted', 'hpwh_adopted', 'net_capacity_diff_amp', 'required_add_capacity_amp', 'upgrade_required', 'upgrade_costs_base', 'location_factor', 'upgrade_costs', 'equiv_annual_cost', 'weighted_cost']

BAU data sample:
   building_id  residential  max_elec_consumption_kwh            timestamp  \
0            1            0                 12.790447  2018-01-17 10:45:00   
1            

### Building Stock Analysis

Let's analyze the building characteristics in the BICEP dataset.

In [10]:
# Compare building characteristics between scenarios
print("\n" + "="*70)
print("BUILDING STOCK COMPARISON")
print("="*70)

print(f"\nBAU Scenario:")
print(f"  Total buildings: {len(bau_res):,}")
print(f"  Residential: {(bau_res['residential'] == 1).sum():,}")
print(f"  Commercial: {(bau_res['residential'] == 0).sum():,}")
print(f"  Average sqft: {bau_res['sqft'].mean():,.0f}")
print(f"  Total sqft: {bau_res['sqft'].sum():,.0f}")

print(f"\nHigh Scenario:")
print(f"  Total buildings: {len(high_res):,}")
print(f"  Residential: {(high_res['residential'] == 1).sum():,}")
print(f"  Commercial: {(high_res['residential'] == 0).sum():,}")
print(f"  Average sqft: {high_res['sqft'].mean():,.0f}")
print(f"  Total sqft: {high_res['sqft'].sum():,.0f}")

print(f"\n" + "="*70)
print("Building datasets contain full analysis results from BICEP,")
print("including capacity requirements and upgrade costs for all technologies.")
print("="*70)


BUILDING STOCK COMPARISON

BAU Scenario:
  Total buildings: 884,596
  Residential: 548,916
  Commercial: 335,680
  Average sqft: 16,348
  Total sqft: 14,461,110,921

High Scenario:
  Total buildings: 884,596
  Residential: 548,916
  Commercial: 335,680
  Average sqft: 16,348
  Total sqft: 14,461,110,921

Building datasets contain full analysis results from BICEP,
including capacity requirements and upgrade costs for all technologies.


In [12]:
# Analyze building distribution by state
print("\nBuildings by Top 10 States (BAU):")
state_dist = bau_res['state'].value_counts().head(10)
for state, count in state_dist.items():
    print(f"  {state}: {count:,}")

print("\nBuildings by Top 10 States (High):")
state_dist_high = high_res['state'].value_counts().head(10)
for state, count in state_dist_high.items():
    print(f"  {state}: {count:,}")

print("\nNote: Both scenarios have identical building distributions.")
print("The difference is in technology adoption assumptions.")


Buildings by Top 10 States (BAU):
  CA: 96,923
  TX: 70,900
  FL: 60,477
  NY: 52,628
  PA: 37,231
  OH: 35,841
  IL: 33,440
  MI: 30,584
  NC: 29,939
  GA: 27,843

Buildings by Top 10 States (High):
  CA: 96,923
  TX: 70,900
  FL: 60,477
  NY: 52,628
  PA: 37,231
  OH: 35,841
  IL: 33,440
  MI: 30,584
  NC: 29,939
  GA: 27,843

Note: Both scenarios have identical building distributions.
The difference is in technology adoption assumptions.


## About BICEP Data

The BICEP results datasets contain pre-calculated infrastructure upgrade requirements for all buildings.

In [13]:
# Show data summary
print("\nBICEP Results Data Summary:")
print(f"  Total records (BAU): {len(bau_res):,}")
print(f"  Total records (High): {len(high_res):,}")
print(f"\nKey data columns in results:")
for col in bau_res.columns:
    print(f"  - {col}")

print(f"\nThe 'upgrade' column indicates whether infrastructure upgrade is required:")
print(f"  BAU upgrades needed: {(bau_res['upgrade'] == 1).sum():,} ({(bau_res['upgrade'] == 1).sum() / len(bau_res) * 100:.1f}%)")
print(f"  High upgrades needed: {(high_res['upgrade'] == 1).sum():,} ({(high_res['upgrade'] == 1).sum() / len(high_res) * 100:.1f}%)")

print(f"\nBoth scenarios have identical building stocks - the difference is in")
print(f"technology adoption assumptions, which drive upgrade requirements.")


BICEP Results Data Summary:
  Total records (BAU): 884,596
  Total records (High): 884,596

Key data columns in results:
  - building_id
  - residential
  - max_elec_consumption_kwh
  - timestamp
  - upgrade
  - state
  - file_path
  - release
  - sqft
  - weight
  - total_units
  - peak_kw
  - peak_amp
  - assumed_volt
  - utilization
  - est_capacity
  - req_capacity
  - installed_capacity
  - spare_capacity
  - hp_peak_diff_kwh
  - hpwh_peak_diff_kwh
  - hp_req_capacity_amp
  - hpwh_req_capacity_amp
  - pv_relative_size
  - pv_size_kw
  - pv_req_capacity_amp
  - total_parking_spaces
  - perc_ev_spaces
  - ev_spaces
  - represented_vehicles
  - ev_req_capacity_amp
  - ev_adopted
  - pv_adopted
  - hp_adopted
  - hpwh_adopted
  - net_capacity_diff_amp
  - required_add_capacity_amp
  - upgrade_required
  - upgrade_costs_base
  - location_factor
  - upgrade_costs
  - equiv_annual_cost
  - weighted_cost

The 'upgrade' column indicates whether infrastructure upgrade is required:
  BAU up

## Using Custom Technology Forecasts

BICEP is designed to work with forecasts from Scout and ReEDS, but you can also integrate custom forecasts. Here's how you would approach this:

### Option 1: Modify Adoption Data Before BICEP Analysis

If you have custom adoption forecasts, you can replace the adoption columns in the residential/commercial DataFrames before analysis.

In [14]:
# Example: Working with custom adoption data
# This shows how you could inject custom forecasts

print("Example: How to integrate custom forecasts with BICEP data")
print("="*60)
print("\nStep 1: Load your custom adoption forecasts")
print("  custom_forecasts = pd.read_csv('my_forecasts.csv')")
print("\nStep 2: Merge with BICEP results by state")
print("  merged = bau_res.merge(custom_forecasts, on='state')")
print("\nStep 3: Modify the adoption columns")
print("  merged['ev_adopted'] = (merged['ev_adoption_rate'] > 0.5).astype(int)")
print("\nStep 4: Use the modified data for analysis")
print("  # Run BICEP analysis on merged data")

Example: How to integrate custom forecasts with BICEP data

Step 1: Load your custom adoption forecasts
  custom_forecasts = pd.read_csv('my_forecasts.csv')

Step 2: Merge with BICEP results by state
  merged = bau_res.merge(custom_forecasts, on='state')

Step 3: Modify the adoption columns
  merged['ev_adopted'] = (merged['ev_adoption_rate'] > 0.5).astype(int)

Step 4: Use the modified data for analysis
  # Run BICEP analysis on merged data


### Option 2: Provide Your Own Forecast File

If you have adoption forecasts in a CSV file, you can merge them with BICEP's building data:

In [15]:
# Example structure for custom adoption data
custom_forecast_structure = pd.DataFrame({
    'state': ['CA', 'CA', 'TX', 'TX'],
    'year': [2030, 2040, 2030, 2040],
    'ev_adoption_rate': [0.25, 0.60, 0.20, 0.50],  # Fraction of buildings
    'hp_adoption_rate': [0.30, 0.70, 0.25, 0.65],
    'hpwh_adoption_rate': [0.20, 0.50, 0.15, 0.45],
    'pv_adoption_rate': [0.15, 0.40, 0.10, 0.35]
})

print("Example format for custom adoption forecasts:")
print(custom_forecast_structure.to_string(index=False))

print("\n" + "="*60)
print("To use custom forecasts:")
print("="*60)
print("1. Prepare data in the format above (state, year, adoption rates)")
print("2. Load your CSV: custom_data = pd.read_csv('your_forecasts.csv')")
print("3. Merge with BICEP building data by state and year")
print("4. Use values in custom_data columns to override adoption columns")
print("5. Run BicepResults on the modified data")

Example format for custom adoption forecasts:
state  year  ev_adoption_rate  hp_adoption_rate  hpwh_adoption_rate  pv_adoption_rate
   CA  2030              0.25              0.30                0.20              0.15
   CA  2040              0.60              0.70                0.50              0.40
   TX  2030              0.20              0.25                0.15              0.10
   TX  2040              0.50              0.65                0.45              0.35

To use custom forecasts:
1. Prepare data in the format above (state, year, adoption rates)
2. Load your CSV: custom_data = pd.read_csv('your_forecasts.csv')
3. Merge with BICEP building data by state and year
4. Use values in custom_data columns to override adoption columns
5. Run BicepResults on the modified data


## Providing Your Own Forecast Data

You don't need complex adoption trajectories! BICEP only requires **two data points**:

1. **Base Year Stock** (e.g., 2020): Current number of installations
2. **End Year Stock** (e.g., 2050): Projected number of installations

BICEP will automatically interpolate between these years to create smooth adoption trajectories.


In [3]:
# Simple example: providing just two data points
import pandas as pd

# Your forecast data - JUST TWO YEARS NEEDED!
your_forecast = pd.DataFrame({
    'state': ['CA', 'TX', 'NY'],
    'technology': ['EV', 'EV', 'EV'],
    'base_year': [2020, 2020, 2020],
    'end_year': [2050, 2050, 2050],
    'base_stock': [500000, 300000, 250000],  # Current installations (2020)
    'end_stock': [12000000, 8500000, 6500000]  # Projected installations (2050)
})

print("=" * 70)
print("MINIMUM DATA REQUIRED - JUST TWO TIME POINTS")
print("=" * 70)
print("\nYour forecast data:")
print(your_forecast.to_string(index=False))

print("\n" + "=" * 70)
print("HOW IT WORKS")
print("=" * 70)
print("\n1. BICEP reads your two data points (base_year and end_year)")
print("2. BICEP creates adoption trajectory by linear interpolation")
print("3. You get infrastructure upgrade costs for your scenario")

# Show how BICEP would interpolate
ca_base = 500000
ca_end = 12000000
ca_years = np.array([2020, 2030, 2040, 2050])
ca_interpolated = np.linspace(ca_base, ca_end, len(ca_years))

print("\nExample: California EV interpolation")
print("-" * 70)
for year, stock in zip(ca_years, ca_interpolated):
    print(f"  {int(year)}: {int(stock):,} installations")

# Visualization
fig = go.Figure()

# Plot user data points
fig.add_trace(go.Scatter(
    x=[2020, 2050], 
    y=[ca_base/1000000, ca_end/1000000],
    mode='markers',
    marker=dict(size=12, color='#d97706'),
    name='Your Data Points',
    hovertemplate='<b>Year: %{x}</b><br>Stock: %{y:.1f}M<extra></extra>'
))

# Plot BICEP interpolation
ca_full_years = np.array([2020, 2025, 2030, 2035, 2040, 2045, 2050])
ca_full_interp = np.linspace(ca_base, ca_end, len(ca_full_years))
fig.add_trace(go.Scatter(
    x=ca_full_years,
    y=ca_full_interp/1000000,
    mode='lines',
    line=dict(color='#f59e0b', width=3),
    name='BICEP Interpolation',
    hovertemplate='<b>Year: %{x}</b><br>Stock: %{y:.1f}M<extra></extra>'
))

fig.update_layout(
    title='<b>Providing Your Own Forecast</b><br><sub>BICEP interpolates between your two data points</sub>',
    xaxis_title='Year',
    yaxis_title='Technology Stock (Millions)',
    hovermode='x unified',
    height=500,
    width=900
)
fig.show()

print("\n" + "=" * 70)
print("THAT'S IT! Just provide two years and BICEP handles the rest.")
print("=" * 70)


MINIMUM DATA REQUIRED - JUST TWO TIME POINTS

Your forecast data:
state technology  base_year  end_year  base_stock  end_stock
   CA         EV       2020      2050      500000   12000000
   TX         EV       2020      2050      300000    8500000
   NY         EV       2020      2050      250000    6500000

HOW IT WORKS

1. BICEP reads your two data points (base_year and end_year)
2. BICEP creates adoption trajectory by linear interpolation
3. You get infrastructure upgrade costs for your scenario

Example: California EV interpolation
----------------------------------------------------------------------
  2020: 500,000 installations
  2030: 4,333,333 installations
  2040: 8,166,666 installations
  2050: 12,000,000 installations



THAT'S IT! Just provide two years and BICEP handles the rest.


## Summary

### Key Takeaways:

1. **BICEP analyzes 4 technologies**: EVs, Heat Pumps, HPWHs, and Solar PV
2. **Two scenarios are available**: BAU (lower adoption) and High (higher adoption)
3. **Technology adoption varies significantly**: The High scenario shows 2-3x higher adoption rates for heating technologies
4. **Custom forecasts can be integrated**: BICEP can work with your own adoption forecasts by merging data by state and year

### Next Steps:

- Explore the [Scenario Comparison](scenario-comparison.ipynb) notebook to see how these adoption differences affect infrastructure costs
- Check the [API Reference](../api-reference.md) for detailed method documentation
- Review the [Custom Distributions](custom-distributions.ipynb) notebook to learn about cost distributions